In [1]:
!pip install -q lightgbm streamlit seaborn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 61.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 42.9 MB/s eta 0:00:00


In [2]:
import os
import zipfile
import pickle
import numpy as np
import pandas as pd
import lightgbm as lgb
import matplotlib.pyplot as plt
import seaborn as sns

from google.colab import drive

from sklearn.model_selection import (train_test_split, StratifiedKFold)

from sklearn.ensemble import RandomForestClassifier

from sklearn.linear_model import LogisticRegression

from sklearn.preprocessing import LabelEncoder

from sklearn.metrics import (roc_auc_score, accuracy_score, brier_score_loss)

from sklearn.calibration import calibration_curve

In [3]:
def entropy(p):
    p = np.clip(p, 1e-9, 1 - 1e-9)
    return -(p * np.log(p) + (1 - p) * np.log(1 - p))


def reduce_mem(df):

    for col in df.columns:

        if df[col].dtype == "object":
            df[col] = df[col].astype("category")

        elif pd.api.types.is_numeric_dtype(df[col]):

            if pd.api.types.is_integer_dtype(df[col]):
                df[col] = pd.to_numeric(
                    df[col],
                    downcast="integer"
                )

            else:
                df[col] = pd.to_numeric(
                    df[col],
                    downcast="float"
                )

    return df


def compute_ece(y_true, y_prob, n_bins=10):

    frac_pos, mean_pred = calibration_curve(
        y_true,
        y_prob,
        n_bins=n_bins,
        strategy="uniform"
    )

    bin_sizes = np.histogram(
        y_prob,
        bins=n_bins,
        range=(0, 1)
    )[0]

    bin_sizes = bin_sizes / bin_sizes.sum()

    n = min(len(frac_pos), len(bin_sizes))

    ece = np.sum(
        np.abs(frac_pos[:n] - mean_pred[:n])
        * bin_sizes[:n]
    )

    return ece

In [4]:
# LOAD DATA

drive.mount('/content/drive')

dataset_path = "/content/drive/MyDrive/ieee-fraud-detection.zip"

extract_path = "/content/ieee_data"

os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(dataset_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)

dfs = {}

for file in os.listdir(extract_path):

    if file.endswith(".csv"):

        name = file.replace(".csv", "")

        dfs[name] = pd.read_csv(
            os.path.join(extract_path, file)
        )

        print(name, dfs[name].shape)

Mounted at /content/drive
train_identity (144233, 41)
test_identity (141907, 41)
train_transaction (590540, 394)
test_transaction (506691, 393)
sample_submission (506691, 2)


In [5]:
# MERGE DATA

train = dfs["train_transaction"].merge(
    dfs["train_identity"],
    on="TransactionID",
    how="left"
)

train.columns = train.columns.str.replace("-", "_")

print("Full Shape:", train.shape)

Full Shape: (590540, 434)


In [6]:
# SAMPLE DATA

train = train.sample(
    n=100000,
    random_state=42
)

print("Sample Shape:", train.shape)

Sample Shape: (100000, 434)


Building Base Model

In [7]:
# PREPARE DATA

target = "isFraud"

X = train.drop(columns=[target])

y = train[target]

X = X.fillna(-999)

X = reduce_mem(X)

In [8]:
# ENCODE CATEGORICALS

cat_cols = X.select_dtypes(
    include="category"
).columns

for col in cat_cols:

    le = LabelEncoder()

    X[col] = le.fit_transform(
        X[col].astype(str)
    )

X = reduce_mem(X)

In [9]:
# TRAIN TEST SPLIT

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

y_train_np = y_train.values
y_test_np = y_test.values

In [10]:
# SAVE EXTRA UI COLUMNS

extra_cols = [
    "card4",
    "card6",
    "DeviceType",
    "ProductCD",
    "TransactionAmt"
]

available_extra_cols = [
    c for c in extra_cols
    if c in train.columns
]

extra_data = train.loc[
    X_test.index,
    available_extra_cols
].copy()

In [11]:
# FEATURE SELECTION

fs_model = lgb.LGBMClassifier(
    n_estimators=100,
    random_state=42
)

fs_model.fit(X_train, y_train_np)

importance = pd.Series(
    fs_model.feature_importances_,
    index=X_train.columns
)

top_features = importance.sort_values(
    ascending=False
).head(150).index

X_train = X_train[top_features]
X_test = X_test[top_features]

print("Selected Features:", len(top_features))

[LightGBM] [Info] Number of positive: 2861, number of negative: 77139
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.504513 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 34966
[LightGBM] [Info] Number of data points in the train set: 80000, number of used features: 432
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.035763 -> initscore=-3.294438
[LightGBM] [Info] Start training from score -3.294438
Selected Features: 150


In [12]:
# ENSEMBLE TRAINING

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

oof_lgb = np.zeros(len(X_train))
oof_rf = np.zeros(len(X_train))
oof_lr = np.zeros(len(X_train))

test_lgb = np.zeros(len(X_test))
test_rf = np.zeros(len(X_test))
test_lr = np.zeros(len(X_test))

for fold, (tr_idx, val_idx) in enumerate(
    skf.split(X_train, y_train_np)
):

    print(f"Fold {fold+1}")

    X_tr = X_train.iloc[tr_idx]
    X_val = X_train.iloc[val_idx]

    y_tr = y_train_np[tr_idx]
    y_val = y_train_np[val_idx]

    # LightGBM
    lgb_model = lgb.LGBMClassifier(
        n_estimators=200,
        learning_rate=0.05,
        random_state=42
    )

    lgb_model.fit(X_tr, y_tr)

    oof_lgb[val_idx] = lgb_model.predict_proba(X_val)[:,1]

    test_lgb += (
        lgb_model.predict_proba(X_test)[:,1]
        / skf.n_splits
    )

    # RandomForest
    rf_model = RandomForestClassifier(
        n_estimators=100,
        n_jobs=-1,
        random_state=42
    )

    rf_model.fit(X_tr, y_tr)

    oof_rf[val_idx] = rf_model.predict_proba(X_val)[:,1]

    test_rf += (
        rf_model.predict_proba(X_test)[:,1]
        / skf.n_splits
    )

    # LogisticRegression
    lr_model = LogisticRegression(
        max_iter=1000,
        random_state=42
    )

    lr_model.fit(X_tr, y_tr)

    oof_lr[val_idx] = lr_model.predict_proba(X_val)[:,1]

    test_lr += (
        lr_model.predict_proba(X_test)[:,1]
        / skf.n_splits
    )

Fold 1
[LightGBM] [Info] Number of positive: 2289, number of negative: 61711
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.088908 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 18576
[LightGBM] [Info] Number of data points in the train set: 64000, number of used features: 150
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.035766 -> initscore=-3.294347
[LightGBM] [Info] Start training from score -3.294347


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Fold 2
[LightGBM] [Info] Number of positive: 2289, number of negative: 61711
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.086901 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 18430
[LightGBM] [Info] Number of data points in the train set: 64000, number of used features: 150
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.035766 -> initscore=-3.294347
[LightGBM] [Info] Start training from score -3.294347


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Fold 3
[LightGBM] [Info] Number of positive: 2289, number of negative: 61711
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.125988 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 18427
[LightGBM] [Info] Number of data points in the train set: 64000, number of used features: 150
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.035766 -> initscore=-3.294347
[LightGBM] [Info] Start training from score -3.294347


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Fold 4
[LightGBM] [Info] Number of positive: 2289, number of negative: 61711
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.089423 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 18712
[LightGBM] [Info] Number of data points in the train set: 64000, number of used features: 150
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.035766 -> initscore=-3.294347
[LightGBM] [Info] Start training from score -3.294347


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Fold 5
[LightGBM] [Info] Number of positive: 2288, number of negative: 61712
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.087874 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 18435
[LightGBM] [Info] Number of data points in the train set: 64000, number of used features: 150
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.035750 -> initscore=-3.294800
[LightGBM] [Info] Start training from score -3.294800


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [13]:
# ENSEMBLE PREDICTIONS

oof_pred = (
    oof_lgb + oof_rf + oof_lr
) / 3

test_pred = (
    test_lgb + test_rf + test_lr
) / 3

base_auc = roc_auc_score(
    y_test_np,
    test_pred
)

base_pred_np = (
    test_pred > 0.5
).astype(int)

base_accuracy = accuracy_score(
    y_test_np,
    base_pred_np
)

print("Base AUC:", base_auc)
print("Base Accuracy:", base_accuracy)

Base AUC: 0.9026587931125137
Base Accuracy: 0.97425


Building Meta Model

In [14]:
# META FEATURES
oof_disagreement = np.std(
    np.vstack([oof_lgb, oof_rf, oof_lr]),
    axis=0
)

test_disagreement = np.std(
    np.vstack([test_lgb, test_rf, test_lr]),
    axis=0
)

train_meta = pd.DataFrame({
    "confidence": oof_pred,
    "entropy": entropy(oof_pred),
    "distance": np.abs(oof_pred - 0.5),
    "missing_count": (
        X_train == -999
    ).sum(axis=1).values,
    "disagreement": oof_disagreement
})

test_meta = pd.DataFrame({
    "confidence": test_pred,
    "entropy": entropy(test_pred),
    "distance": np.abs(test_pred - 0.5),
    "missing_count": (
        X_test == -999
    ).sum(axis=1).values,
    "disagreement": test_disagreement
})

train_meta_target = (
    y_train_np != (
        oof_pred > 0.5
    ).astype(int)
).astype(int)

test_meta_target = (
    y_test_np != base_pred_np
).astype(int)


In [15]:
# META MODEL

meta_model = lgb.LGBMClassifier(
    n_estimators=200,
    random_state=42
)

meta_model.fit(
    train_meta,
    train_meta_target
)

meta_test_prob = meta_model.predict_proba(
    test_meta
)[:,1]

meta_auc = roc_auc_score(
    test_meta_target,
    meta_test_prob
)

meta_brier = brier_score_loss(
    test_meta_target,
    meta_test_prob
)

meta_ece = compute_ece(
    test_meta_target,
    meta_test_prob
)

print("Meta AUC:", meta_auc)

[LightGBM] [Info] Number of positive: 2113, number of negative: 77887
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.017907 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1090
[LightGBM] [Info] Number of data points in the train set: 80000, number of used features: 5
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.026412 -> initscore=-3.607150
[LightGBM] [Info] Start training from score -3.607150
Meta AUC: 0.8608046518232846


In [16]:
# RELIABILITY

reliability_score = 1 - meta_test_prob

In [17]:
# FINAL RESULTS

results = extra_data.copy()

results["y_true"] = y_test_np
results["y_pred"] = base_pred_np
results["y_prob"] = test_pred
results["error_prob"] = meta_test_prob
results["reliability_score"] = reliability_score

results["error"] = (
    results["y_true"] != results["y_pred"]
).astype(int)


In [18]:
# THRESHOLD SWEEP

thresholds = np.arange(
    0.40,
    0.96,
    0.01
)

coverages = []
accs = []

for t in thresholds:

    mask = (
        results["reliability_score"] > t
    )

    if mask.sum() == 0:

        coverages.append(0)
        accs.append(np.nan)

        continue

    coverages.append(mask.mean())

    accs.append(
        accuracy_score(
            results.loc[mask, "y_true"],
            results.loc[mask, "y_pred"]
        )
    )

sweep_df = pd.DataFrame({
    "threshold": thresholds,
    "coverage": coverages,
    "accuracy": accs
})

In [19]:
# SUMMARY METRICS

summary_metrics = {
    "base_auc": base_auc,
    "base_accuracy": base_accuracy,
    "meta_auc": meta_auc,
    "meta_brier": meta_brier,
    "meta_ece": meta_ece
}

In [20]:
# SAVE DASHBOARD DATA

with open("dashboard_data.pkl", "wb") as f:

    pickle.dump({
        "results": results,
        "sweep_df": sweep_df,
        "summary_metrics": summary_metrics,
        "meta_test_prob": meta_test_prob,
        "test_meta_target": test_meta_target
    }, f)

print("dashboard_data.pkl saved")

dashboard_data.pkl saved


Building Streamlit Dashboard

In [21]:
# =========================================================
# AI RELIABILITYGUARD - WITH FRAUD PREDICTION
# =========================================================

%%writefile streamlit_app.py

import streamlit as st
import pickle
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

# =========================================================
# PAGE CONFIG
# =========================================================

st.set_page_config(
    page_title="AI ReliabilityGuard",
    layout="wide",
    initial_sidebar_state="expanded"
)

st.set_option('client.showErrorDetails', False)

# =========================================================
# CUSTOM CSS — Dark Fintech Aesthetic
# =========================================================

st.markdown("""
<style>
@import url('https://fonts.googleapis.com/css2?family=Space+Mono:wght@400;700&family=DM+Sans:wght@300;400;600;700&display=swap');

/* ---------- global ---------- */
html, body, [class*="css"] {
    font-family: 'DM Sans', sans-serif;
    background-color: #0a0e1a;
    color: #e2e8f0;
}

.stApp { background-color: #0a0e1a; }

/* ---------- title ---------- */
.main-title {
    font-family: 'Space Mono', monospace;
    font-size: 2rem;
    font-weight: 700;
    letter-spacing: -1px;
    background: linear-gradient(135deg, #38bdf8 0%, #818cf8 100%);
    -webkit-background-clip: text;
    -webkit-text-fill-color: transparent;
    margin-bottom: 0;
}
.sub-title {
    color: #64748b;
    font-size: 0.85rem;
    letter-spacing: 0.15em;
    text-transform: uppercase;
    margin-top: 2px;
    margin-bottom: 1.5rem;
}

/* ---------- metric cards ---------- */
[data-testid="metric-container"] {
    background: #111827;
    border: 1px solid #1e293b;
    border-radius: 12px;
    padding: 16px 20px;
    box-shadow: 0 4px 24px rgba(0,0,0,0.4);
}
[data-testid="metric-container"] [data-testid="stMetricLabel"] {
    color: #64748b !important;
    font-size: 0.75rem !important;
    text-transform: uppercase;
    letter-spacing: 0.1em;
}
[data-testid="metric-container"] [data-testid="stMetricValue"] {
    color: #f1f5f9 !important;
    font-family: 'Space Mono', monospace;
    font-size: 1.5rem !important;
}

/* ---------- section headers ---------- */
.section-head {
    font-family: 'Space Mono', monospace;
    font-size: 0.8rem;
    color: #38bdf8;
    text-transform: uppercase;
    letter-spacing: 0.2em;
    border-left: 3px solid #38bdf8;
    padding-left: 10px;
    margin: 1.5rem 0 0.8rem;
}

/* ---------- fraud verdict cards ---------- */
.verdict-fraud {
    background: linear-gradient(135deg, #7f1d1d 0%, #450a0a 100%);
    border: 1px solid #ef4444;
    border-radius: 14px;
    padding: 20px 24px;
    text-align: center;
}
.verdict-safe {
    background: linear-gradient(135deg, #052e16 0%, #064e3b 100%);
    border: 1px solid #22c55e;
    border-radius: 14px;
    padding: 20px 24px;
    text-align: center;
}
.verdict-label {
    font-family: 'Space Mono', monospace;
    font-size: 1.4rem;
    font-weight: 700;
    margin: 0;
}
.verdict-sub {
    font-size: 0.8rem;
    color: #94a3b8;
    margin-top: 4px;
}

/* ---------- sidebar ---------- */
[data-testid="stSidebar"] {
    background-color: #0f172a !important;
    border-right: 1px solid #1e293b;
}
[data-testid="stSidebar"] * { color: #cbd5e1 !important; }

/* ---------- dataframe ---------- */
[data-testid="stDataFrame"] {
    border: 1px solid #1e293b;
    border-radius: 10px;
    overflow: hidden;
}

/* ---------- alerts ---------- */
.stAlert { border-radius: 10px !important; }

/* ---------- divider ---------- */
hr { border-color: #1e293b !important; }

/* ---------- plot backgrounds ---------- */
.plot-wrap {
    background: #111827;
    border: 1px solid #1e293b;
    border-radius: 12px;
    padding: 12px;
}
</style>
""", unsafe_allow_html=True)

# =========================================================
# LOAD DATA
# =========================================================

@st.cache_data
def load_data():
    with open("dashboard_data.pkl", "rb") as f:
        return pickle.load(f)

data = load_data()
results    = data["results"]
sweep_df   = data["sweep_df"]
summary_metrics = data["summary_metrics"]

if len(results) > 30000:
    results = results.sample(30000, random_state=42)

# =========================================================
# FRAUD PREDICTION LOGIC
# =========================================================

def predict_fraud(row):
    if "isFraud" in row:
        return row["isFraud"]
    if "fraud" in row:
        return row["fraud"]
    score = row.get("reliability_score", 0.5)
    err   = row.get("error", 0.0)
    return 1 if (score < 0.5 and err > 0.2) else 0

results["fraud_prediction"] = results.apply(predict_fraud, axis=1)

# =========================================================
# MATPLOTLIB STYLE — dark theme
# =========================================================

plt.rcParams.update({
    "figure.facecolor":  "#111827",
    "axes.facecolor":    "#111827",
    "axes.edgecolor":    "#1e293b",
    "axes.labelcolor":   "#94a3b8",
    "xtick.color":       "#64748b",
    "ytick.color":       "#64748b",
    "text.color":        "#e2e8f0",
    "grid.color":        "#1e293b",
    "grid.linestyle":    "--",
    "grid.alpha":        0.5,
    "font.family":       "monospace",
})

# =========================================================
# TITLE
# =========================================================

st.markdown('<p class="main-title">🛡️ AI ReliabilityGuard</p>', unsafe_allow_html=True)
st.markdown('<p class="sub-title">Failure Forecasting · Fraud Detection · Reliability Scoring</p>', unsafe_allow_html=True)

# =========================================================
# SIDEBAR FILTERS
# =========================================================

st.sidebar.markdown("## 🔍 Filters")

selected_card   = None
selected_card6  = None
selected_device = None
selected_product = None
selected_amt    = None

if "card4" in results.columns:
    selected_card = st.sidebar.selectbox(
        "Card Network", sorted(results["card4"].dropna().unique())
    )

if "card6" in results.columns:
    selected_card6 = st.sidebar.selectbox(
        "Card Type", sorted(results["card6"].dropna().unique())
    )

if "DeviceType" in results.columns:
    selected_device = st.sidebar.selectbox(
        "Device Type", sorted(results["DeviceType"].dropna().unique())
    )

if "ProductCD" in results.columns:
    selected_product = st.sidebar.selectbox(
        "Product Type", sorted(results["ProductCD"].dropna().unique())
    )

# ── Transaction amount capped at 1000 ──────────────────
if "TransactionAmt" in results.columns:
    min_amt = int(results["TransactionAmt"].min())
    max_amt = min(1000, int(results["TransactionAmt"].max()))   # ← capped at 1000
    default_val = int((min_amt + max_amt) / 2)

    selected_amt = st.sidebar.slider(
        "Min Transaction Amount ($)",
        min_value=min_amt,
        max_value=max_amt,
        value=default_val,
        step=10
    )

st.sidebar.markdown("---")
st.sidebar.markdown("### 🔮 Single-Transaction Check")
st.sidebar.markdown("Paste values below to get an instant fraud verdict for one transaction.")

manual_reliability = st.sidebar.slider("Reliability Score", 0.0, 1.0, 0.75, 0.01)
manual_error       = st.sidebar.slider("Error Rate",        0.0, 1.0, 0.10, 0.01)
manual_amt         = st.sidebar.number_input("Transaction Amount ($)", 0, 1000, 150, step=10)

# =========================================================
# FILTERING
# =========================================================

filtered = results.copy()

if selected_card:    filtered = filtered[filtered["card4"]      == selected_card]
if selected_card6:   filtered = filtered[filtered["card6"]      == selected_card6]
if selected_device:  filtered = filtered[filtered["DeviceType"] == selected_device]
if selected_product: filtered = filtered[filtered["ProductCD"]  == selected_product]
if selected_amt is not None and "TransactionAmt" in filtered.columns:
    filtered = filtered[filtered["TransactionAmt"] >= selected_amt]

# =========================================================
# SINGLE-TRANSACTION FRAUD VERDICT  (sidebar → main)
# =========================================================

st.markdown('<p class="section-head">🔮 Single-Transaction Fraud Verdict</p>', unsafe_allow_html=True)

single_fraud = 1 if (manual_reliability < 0.5 and manual_error > 0.2) else 0
risk_pct     = round((1 - manual_reliability) * 60 + manual_error * 40, 1)

vcol1, vcol2, vcol3 = st.columns([1, 1, 1])

with vcol1:
    if single_fraud == 1:
        st.markdown(f"""
        <div class="verdict-fraud">
            <p class="verdict-label">🔴 FRAUD</p>
            <p class="verdict-sub">Transaction flagged as high-risk</p>
        </div>""", unsafe_allow_html=True)
    else:
        st.markdown(f"""
        <div class="verdict-safe">
            <p class="verdict-label">🟢 LEGITIMATE</p>
            <p class="verdict-sub">Transaction appears trustworthy</p>
        </div>""", unsafe_allow_html=True)

with vcol2:
    st.metric("Reliability Score", f"{manual_reliability:.0%}")
    st.metric("Error Rate",        f"{manual_error:.0%}")

with vcol3:
    st.metric("Transaction Amt",   f"${manual_amt}")
    st.metric("Composite Risk",    f"{risk_pct:.1f}%")

st.markdown("---")

# =========================================================
# AGGREGATE METRICS
# =========================================================

st.markdown('<p class="section-head">📊 Aggregate Dashboard</p>', unsafe_allow_html=True)

if len(filtered) == 0:
    st.warning("⚠️ No transactions match your current filters.")
else:
    avg_rel    = filtered["reliability_score"].mean()
    err_rate   = filtered["error"].mean()
    fraud_rate = filtered["fraud_prediction"].mean()

    m1, m2, m3, m4 = st.columns(4)
    m1.metric("Transactions",     f"{len(filtered):,}")
    m2.metric("Avg Reliability",  f"{avg_rel:.1%}")
    m3.metric("Error Rate",       f"{err_rate:.1%}")
    m4.metric("Fraud Rate",       f"{fraud_rate:.1%}")

    # ── Risk banner ──────────────────────────────────────
    if fraud_rate > 0.5:
        st.error("🔴 HIGH FRAUD RISK — Immediate review recommended.")
    elif fraud_rate > 0.2:
        st.warning("🟡 MODERATE FRAUD RISK — Monitor closely.")
    else:
        st.success("🟢 LOW FRAUD RISK — System operating normally.")

    # =====================================================
    # CHARTS — side by side, compact
    # =====================================================

    st.markdown('<p class="section-head">📈 Visual Analytics</p>', unsafe_allow_html=True)

    chart_col1, chart_col2 = st.columns(2)

    # -- Chart 1: Reliability distribution ----------------
    with chart_col1:
        fig1, ax1 = plt.subplots(figsize=(4.5, 2.8))
        ax1.hist(
            filtered["reliability_score"],
            bins=25,
            color="#38bdf8",
            edgecolor="#0a0e1a",
            alpha=0.85
        )
        ax1.axvline(avg_rel, color="#f59e0b", linewidth=1.5, linestyle="--", label=f"Mean {avg_rel:.2f}")
        ax1.set_xlabel("Reliability Score", fontsize=8)
        ax1.set_ylabel("Count", fontsize=8)
        ax1.set_title("Reliability Distribution", fontsize=9, color="#e2e8f0", pad=6)
        ax1.legend(fontsize=7, facecolor="#111827", edgecolor="#1e293b", labelcolor="#e2e8f0")
        ax1.tick_params(labelsize=7)
        fig1.tight_layout(pad=0.8)
        st.pyplot(fig1, use_container_width=True)
        plt.close(fig1)

    # -- Chart 2: Fraud vs Legitimate count ---------------
    with chart_col2:
        fraud_counts = filtered["fraud_prediction"].value_counts().sort_index()
        labels = ["Legitimate", "Fraud"] if len(fraud_counts) == 2 else (
            ["Legitimate"] if fraud_counts.index[0] == 0 else ["Fraud"]
        )
        colors = ["#22c55e", "#ef4444"][:len(fraud_counts)]

        fig2, ax2 = plt.subplots(figsize=(4.5, 2.8))
        bars = ax2.bar(
            labels if len(labels) == len(fraud_counts) else [str(i) for i in fraud_counts.index],
            fraud_counts.values,
            color=colors,
            edgecolor="#0a0e1a",
            width=0.45
        )
        for bar, val in zip(bars, fraud_counts.values):
            ax2.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height() + max(fraud_counts.values) * 0.02,
                f"{val:,}", ha="center", va="bottom", fontsize=8, color="#e2e8f0"
            )
        ax2.set_ylabel("Transactions", fontsize=8)
        ax2.set_title("Fraud vs Legitimate", fontsize=9, color="#e2e8f0", pad=6)
        ax2.tick_params(labelsize=7)
        fig2.tight_layout(pad=0.8)
        st.pyplot(fig2, use_container_width=True)
        plt.close(fig2)

    # ── Row 2 charts ─────────────────────────────────────
    chart_col3, chart_col4 = st.columns(2)

    # -- Chart 3: Error distribution ----------------------
    with chart_col3:
        fig3, ax3 = plt.subplots(figsize=(4.5, 2.8))
        ax3.hist(
            filtered["error"],
            bins=25,
            color="#818cf8",
            edgecolor="#0a0e1a",
            alpha=0.85
        )
        ax3.axvline(err_rate, color="#f59e0b", linewidth=1.5, linestyle="--", label=f"Mean {err_rate:.2f}")
        ax3.set_xlabel("Error Rate", fontsize=8)
        ax3.set_ylabel("Count", fontsize=8)
        ax3.set_title("Error Distribution", fontsize=9, color="#e2e8f0", pad=6)
        ax3.legend(fontsize=7, facecolor="#111827", edgecolor="#1e293b", labelcolor="#e2e8f0")
        ax3.tick_params(labelsize=7)
        fig3.tight_layout(pad=0.8)
        st.pyplot(fig3, use_container_width=True)
        plt.close(fig3)

    # -- Chart 4: Reliability vs Error scatter ------------
    with chart_col4:
        sample_s = filtered.sample(min(2000, len(filtered)), random_state=1)
        fraud_mask = sample_s["fraud_prediction"] == 1

        fig4, ax4 = plt.subplots(figsize=(4.5, 2.8))
        ax4.scatter(
            sample_s.loc[~fraud_mask, "reliability_score"],
            sample_s.loc[~fraud_mask, "error"],
            s=8, alpha=0.4, color="#22c55e", label="Legitimate"
        )
        ax4.scatter(
            sample_s.loc[fraud_mask, "reliability_score"],
            sample_s.loc[fraud_mask, "error"],
            s=8, alpha=0.6, color="#ef4444", label="Fraud"
        )
        ax4.axvline(0.5, color="#f59e0b", linewidth=1, linestyle="--", alpha=0.7)
        ax4.axhline(0.2, color="#f59e0b", linewidth=1, linestyle="--", alpha=0.7)
        ax4.set_xlabel("Reliability Score", fontsize=8)
        ax4.set_ylabel("Error", fontsize=8)
        ax4.set_title("Reliability vs Error (Fraud Map)", fontsize=9, color="#e2e8f0", pad=6)
        ax4.legend(fontsize=7, facecolor="#111827", edgecolor="#1e293b", labelcolor="#e2e8f0", markerscale=2)
        ax4.tick_params(labelsize=7)
        fig4.tight_layout(pad=0.8)
        st.pyplot(fig4, use_container_width=True)
        plt.close(fig4)

    # =====================================================
    # TRANSACTION TABLE
    # =====================================================

    st.markdown('<p class="section-head">📋 Transaction Risk Table</p>', unsafe_allow_html=True)

    show_cols = [c for c in [
        "TransactionAmt", "card4", "card6", "DeviceType",
        "reliability_score", "error", "fraud_prediction"
    ] if c in filtered.columns]

    display_df = filtered[show_cols].head(30).copy()
    if "fraud_prediction" in display_df.columns:
        display_df["fraud_prediction"] = display_df["fraud_prediction"].map({0: "✅ Legitimate", 1: "🔴 Fraud"})

    st.dataframe(display_df, use_container_width=True, height=300)

# =========================================================
# FOOTER
# =========================================================

st.markdown("---")
st.markdown("""
<div style="text-align:center; color:#334155; font-size:0.75rem; font-family:'Space Mono',monospace;">
    AI ReliabilityGuard &nbsp;·&nbsp; Failure Forecasting &nbsp;·&nbsp; Fraud Detection &nbsp;·&nbsp; Capstone — Data 606
</div>
""", unsafe_allow_html=True)

Writing streamlit_app.py


In [22]:
!streamlit run streamlit_app.py --server.port 8501 --server.address 0.0.0.0 > streamlit.log 2>&1 &

In [23]:
import time
time.sleep(15)

In [24]:
!ps -ef | grep streamlit

root        7862       1 19 01:02 ?        00:00:02 /usr/bin/python3 /usr/local/bin/streamlit run streamlit_app.py --server.port 8501 --server.address 0.0.0.0
root        7930    5366  0 01:02 ?        00:00:00 /bin/bash -c ps -ef | grep streamlit
root        7932    7930  0 01:02 ?        00:00:00 grep streamlit


In [25]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

In [26]:
!ls

cloudflared	    drive      sample_data	 streamlit.log
dashboard_data.pkl  ieee_data  streamlit_app.py


In [28]:
!curl http://localhost:8501

<!--
 Copyright (c) Streamlit Inc. (2018-2022) Snowflake Inc. (2022-2026)

 Licensed under the Apache License, Version 2.0 (the "License");
 you may not use this file except in compliance with the License.
 You may obtain a copy of the License at

     http://www.apache.org/licenses/LICENSE-2.0

 Unless required by applicable law or agreed to in writing, software
 distributed under the License is distributed on an "AS IS" BASIS,
 WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
 See the License for the specific language governing permissions and
 limitations under the License.
-->

<!DOCTYPE html>
<html lang="en">
  <head>
    <meta charset="UTF-8" />
    <meta
      name="viewport"
      content="width=device-width, initial-scale=1, shrink-to-fit=no"
    />
    <link rel="shortcut icon" href="./favicon.png" />
    <link
      rel="preload"
      href="./static/media/SourceSansVF-Upright.ttf.BsWL4Kly.woff2"
      as="font"
      type="font/woff2"
      crossorig

In [ ]:
!./cloudflared tunnel --url http://localhost:8501

2026-05-18T01:02:36Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-05-18T01:02:36Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-05-18T01:02:40Z INF +--------------------------------------------------------------------------------------------+
2026-05-18T01:02:40Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-05-18T01:02:40Z INF |  https://himself-burst-domains-envelope.trycloudflare.